# SiPEKA — Notebook Persiapan Data (Data Preparation)

**Penulisan Ilmiah:** Optimasi XGBoost dengan SMOTE untuk Prediksi Risiko Preeklampsia (SiPEKA)  
**Tahap CRISP-DM:** *Data Preparation*

Notebook ini mengubah **data mentah** laporan kunjungan ibu hamil (ANC) Puskesmas Cipayung menjadi **dataset primer final** yang rapi dan siap dimodelkan.

### Tantangan data mentah
Berkas laporan Puskesmas **sangat sulit dibaca oleh program** karena:
1. **Header bertingkat & ter-merge.** Kolom `No` di-merge dari baris ke-7 sampai ke-10 (A7:A10), sedangkan kolom pemeriksaan di-merge (mis. B8:B10) dengan baris di atasnya menjelaskan *grup* datanya. Jadi satu kolom bisa punya 2–3 baris judul.
2. **Data baru dimulai di bawah blok header** (bukan baris pertama).
3. **Kolomnya sangat banyak (89 kolom)**, padahal yang dibutuhkan hanya sedikit.
4. **Format nilai tidak seragam:** umur berupa teks (`'32 Thn 6 Bln 16 Hr'`), tekanan darah tergabung (`'127/75'`), status pemeriksaan berupa huruf `Y/T`.

### Solusi
Notebook ini **tidak** mengandalkan posisi kolom yang tetap (rapuh), melainkan:
- **mendeteksi otomatis baris awal data**, dan
- **mencari kolom berdasarkan NAMA header** (menggabungkan seluruh baris header yang ter-merge per kolom).

Alur lengkap: **Baca mentah → Deteksi kolom by-name → Ekstraksi & transformasi → Dataset rapi (masih tanpa label) → Klasifikasi rule-based (pemberian label) → Pembersihan → Simpan.**


## 1. Import Library


In [ ]:
import pandas as pd
import numpy as np
import re, glob, os
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 30)
print('Library siap.')

## 2. Unggah 4 Berkas Bulanan Sekaligus

Jalankan sel di bawah lalu pilih **keempat berkas** Excel (Januari–April) sekaligus.
Bila notebook dijalankan secara lokal, cukup letakkan keempat berkas di folder yang sama.


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()   # pilih 4 berkas sekaligus
    BERKAS = sorted(uploaded.keys())
except Exception:
    # mode lokal: ambil semua berkas laporan di folder ini
    BERKAS = sorted(glob.glob('*Laporan Kunjungan Ibu Hamil*.xlsx'))

print('Berkas yang akan diproses:')
for b in BERKAS: print('  -', b)
assert len(BERKAS) >= 1, 'Tidak ada berkas ditemukan. Unggah dulu berkasnya.'

## 3. Melihat Betapa Berantakannya Data Mentah

Contoh 12 baris pertama dari salah satu berkas — perhatikan blok header bertingkat dan data yang baru muncul di bawahnya.


In [ ]:
contoh = pd.read_excel(BERKAS[0], header=None)
print('Dimensi berkas mentah:', contoh.shape, '(baris x kolom)')
contoh.iloc[:12, :16]

## 4. Fungsi Pembaca Cerdas (Deteksi Kolom by-Name)

Tiga fungsi inti agar pembacaan tahan terhadap header ter-merge:
- **`temukan_mulai`** — mencari baris pertama data (baris di mana kolom `No` berisi angka).
- **`header_gabungan`** — untuk tiap kolom, menggabungkan semua potongan judul dari baris-baris header (menangani sel ter-merge) menjadi satu teks.
- **`cari`** — menemukan indeks kolom berdasarkan **kata kunci nama**, bukan posisi tetap.


In [ ]:
def temukan_mulai(raw):
    """Baris pertama data = baris pertama yang kolom-0 nya berupa angka (No urut)."""
    for i in range(min(40, len(raw))):
        v = raw.iloc[i, 0]
        if pd.notna(v) and str(v).strip().isdigit():
            return i
    raise ValueError('Baris awal data tidak ditemukan.')

def header_gabungan(raw, mulai):
    """Gabungkan seluruh baris header (0..mulai-1) per kolom menjadi satu teks kecil."""
    hdr = {}
    for c in range(raw.shape[1]):
        frag = [str(raw.iloc[r, c]).strip() for r in range(mulai) if pd.notna(raw.iloc[r, c])]
        hdr[c] = ' | '.join(frag).lower()
    return hdr

def cari(hdr, kata_kunci, hindari=()):
    """Kembalikan indeks kolom yang teks headernya memuat semua kata_kunci (dan tidak memuat kata 'hindari')."""
    for c, txt in hdr.items():
        if all(k.lower() in txt for k in kata_kunci) and not any(h.lower() in txt for h in hindari):
            return c
    return None

# peta variabel yang dibutuhkan -> (kata kunci nama, kata yang dihindari)
TARGET = {
    'Usia_Tahun'            : (['umur'],           ['usia kehamilan', 'lahir']),
    'Usia_Kehamilan_Minggu' : (['usia kehamilan'], []),
    'BB'                    : (['bb(kg)'],         []),
    'TB'                    : (['tinggi badan'],   []),
    'TD'                    : (['td(mmhg)'],       []),
    'LILA_cm'               : (['lila'],           []),
    'Protein_Uria'          : (['protein uria'],   []),
    'Gula_Darah'            : (['gula darah'],     []),
}
print('Fungsi pembaca siap.')

## 5. Fungsi Transformasi Nilai

Membersihkan format nilai yang tidak seragam menjadi numerik/biner yang benar.


In [ ]:
def parse_umur(x):
    if pd.isna(x): return np.nan
    m = re.search(r'(\d+)\s*Thn', str(x))
    return float(m.group(1)) if m else np.nan

def pecah_td(x):
    if pd.isna(x): return (np.nan, np.nan)
    m = re.match(r'\s*(\d+)\s*/\s*(\d+)', str(x))
    return (float(m.group(1)), float(m.group(2))) if m else (np.nan, np.nan)

def ya_tidak(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().upper()
    return 1 if s == 'Y' else (0 if s == 'T' else np.nan)

def ke_angka(x):
    try: return float(str(x).replace(',', '.'))
    except: return np.nan

print('Fungsi transformasi siap.')

## 6. Ekstraksi & Penggabungan Seluruh Berkas

Untuk tiap berkas: deteksi baris awal → petakan kolom by-name → ekstrak & transformasi tiap baris. Variabel turunan dihitung: **MAP** = (Sistolik + 2×Diastolik)/3 dan **IMT** = BB / (TB/100)².


In [ ]:
baris = []
for berkas in BERKAS:
    raw = pd.read_excel(berkas, header=None)
    mulai = temukan_mulai(raw)
    hdr = header_gabungan(raw, mulai)
    kol = {nama: cari(hdr, kw, hindari=hv) for nama, (kw, hv) in TARGET.items()}
    hilang = [k for k, v in kol.items() if v is None]
    if hilang:
        print(f'  PERINGATAN {os.path.basename(berkas)}: kolom tak ditemukan -> {hilang}')
    data = raw.iloc[mulai:]
    n = 0
    for _, r in data.iterrows():
        if pd.isna(r[kol['TD']]) and pd.isna(r[kol['Usia_Tahun']]):
            continue
        sys, dia = pecah_td(r[kol['TD']])
        bb, tb = ke_angka(r[kol['BB']]), ke_angka(r[kol['TB']])
        imt = round(bb / ((tb/100)**2), 1) if (bb and tb and tb > 0) else np.nan
        mapv = round((sys + 2*dia)/3, 0) if not (np.isnan(sys) or np.isnan(dia)) else np.nan
        baris.append(dict(
            TD_Sistolik=sys, TD_Diastolik=dia, MAP=mapv,
            Protein_Uria=ya_tidak(r[kol['Protein_Uria']]),
            Usia_Tahun=parse_umur(r[kol['Usia_Tahun']]),
            Usia_Kehamilan_Minggu=ke_angka(r[kol['Usia_Kehamilan_Minggu']]),
            IMT=imt, LILA_cm=ke_angka(r[kol['LILA_cm']]),
            Gula_Darah=ya_tidak(r[kol['Gula_Darah']]),
        ))
        n += 1
    print(f'  {os.path.basename(berkas)}: mulai baris idx {mulai}, terbaca {n} baris')

df = pd.DataFrame(baris)
print('\nTotal baris mentah tergabung:', len(df))

## 7. Dataset Rapi — Namun Masih **Tanpa Label**

Setelah dirapikan, data sudah terstruktur, tetapi **belum memiliki kolom target/label** (unlabeled). Tidak ada kolom diagnosis dokter pada data mentah, sehingga label harus dibentuk sendiri pada tahap berikutnya.


In [ ]:
print('Kolom saat ini (belum ada label):')
print(list(df.columns))
print('\nApakah ada kolom label/target?', 'Risiko_Preeklampsia' in df.columns)
df.head()

## 8. Pembersihan Data (Complete-Case)

Baris tanpa nilai **tekanan darah** atau **proteinuria** dikeluarkan, karena keduanya menjadi penentu utama label pada tahap klasifikasi.


In [ ]:
sebelum = len(df)
df = df.dropna(subset=['TD_Sistolik', 'TD_Diastolik', 'Protein_Uria']).reset_index(drop=True)
df['Protein_Uria'] = df['Protein_Uria'].astype(int)
print(f'Baris sebelum pembersihan : {sebelum}')
print(f'Baris setelah pembersihan : {len(df)}')
print(f'Baris dikeluarkan         : {sebelum - len(df)}')

## 9. Klasifikasi Rule-Based (Pemberian Label)

Karena dataset masih *unlabeled*, label dibentuk secara **terprogram** menggunakan aturan klinis (mengacu PNPK Kemenkes + pengelompokan fitur hasil validasi tenaga kebidanan):

> Data diberi label **berisiko (1)** apabila **(Sistolik ≥ 130 mmHg ATAU Diastolik ≥ 85 mmHg)** **DAN** **proteinuria positif**; selain itu **tidak berisiko (0)**.

Pendekatan ini objektif, konsisten, dan dapat direplikasi.


In [ ]:
df['Risiko_Preeklampsia'] = (
    ((df['TD_Sistolik'] >= 130) | (df['TD_Diastolik'] >= 85)) & (df['Protein_Uria'] == 1)
).astype(int)

dist = df['Risiko_Preeklampsia'].value_counts().sort_index()
print('Distribusi label setelah klasifikasi rule-based:')
print('  Tidak berisiko (0):', int(dist.get(0, 0)))
print('  Berisiko       (1):', int(dist.get(1, 0)))
print(f'  Rasio ketidakseimbangan ~ 1:{dist.get(0,0)/max(dist.get(1,1),1):.1f}')

## 10. Ringkasan & Visualisasi


In [ ]:
print('Dimensi dataset final:', df.shape)
print('\nNilai kosong per kolom:')
print(df.isna().sum())
df.describe().round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df['Risiko_Preeklampsia'].value_counts().sort_index().plot(
    kind='bar', ax=ax[0], color=['#4C72B0', '#C44E52'])
ax[0].set_title('Distribusi Kelas (0=Tidak, 1=Berisiko)')
ax[0].set_xticklabels(['Tidak berisiko', 'Berisiko'], rotation=0)
ax[0].set_ylabel('Jumlah')
ax[1].hist(df['MAP'].dropna(), bins=25, color='#55A868', edgecolor='white')
ax[1].set_title('Sebaran MAP (mmHg)')
ax[1].set_xlabel('MAP')
plt.tight_layout(); plt.show()

## 11. Menyimpan Dataset Final

Disimpan sebagai `.csv` dan `.xlsx`, menjadi masukan notebook pemodelan (`SiPEKA_Modeling.ipynb`).


In [ ]:
URUTAN = ['TD_Sistolik','TD_Diastolik','MAP','Protein_Uria','Usia_Tahun',
          'Usia_Kehamilan_Minggu','IMT','LILA_cm','Gula_Darah','Risiko_Preeklampsia']
df = df[URUTAN]
df.to_csv('Dataset_Primer_Final.csv', index=False)
df.to_excel('Dataset_Primer_Final.xlsx', index=False)
print('Tersimpan: Dataset_Primer_Final.csv & .xlsx')
print('Baris:', len(df), '| Fitur:', len(URUTAN)-1, '| Label: Risiko_Preeklampsia')

# (Colab) unduh otomatis:
# from google.colab import files
# files.download('Dataset_Primer_Final.csv')

---
### Catatan
Notebook ini adalah tahap **Data Preparation** dari alur CRISP-DM. Pemodelan (pembagian data, imputasi, penskalaan, SMOTE, pelatihan XGBoost) dilakukan pada notebook terpisah agar tiap tahap terdokumentasi jelas dan bebas kebocoran data (*data leakage*).
